In [1]:
%pip install keras
%pip install tensorflow

In [2]:


import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle


In [5]:
data_path = '.'
fname = 'dataset'

X_data = np.load(f'{data_path}/{fname}_X.npy', allow_pickle=True).item()  # shape: (N, T)
y_data = np.load(f'{data_path}/{fname}_y.npy', allow_pickle=True).item()  # shape: (N,)

print(X_data.train.shape)
print(y_data.train.shape)



(9088, 500, 3)
(9088,)


In [6]:
X_train = X_data.train
X_val   = X_data.val
X_test  = X_data.test

y_train_raw = y_data.train
y_val_raw   = y_data.val
y_test_raw  = y_data.test

from sklearn.preprocessing import LabelEncoder
import joblib

# Fit on the full set of labels
label_encoder = LabelEncoder()
y_all_raw = np.concatenate([y_train_raw, y_val_raw, y_test_raw])
y_all_enc = label_encoder.fit_transform(y_all_raw)

# Save it
joblib.dump(label_encoder, f'{data_path}/label_encoder.pkl')

# Encode each split
y_train = label_encoder.transform(y_train_raw)
y_val   = label_encoder.transform(y_val_raw)
y_test  = label_encoder.transform(y_test_raw)



In [7]:
# X shape: (N, T, 3)
X_train_scaled = np.zeros_like(X_train)
X_val_scaled = np.zeros_like(X_val)
X_test_scaled = np.zeros_like(X_test)

scalers = []

for i in range(X_train.shape[2]):  # loop over channels: 0=x, 1=y, 2=z
    scaler = StandardScaler()
    scaler.fit(X_train[:, :, i])  # fit on (N, T) for this channel

    X_train_scaled[:, :, i] = scaler.transform(X_train[:, :, i])
    X_val_scaled[:, :, i]   = scaler.transform(X_val[:, :, i])
    X_test_scaled[:, :, i]  = scaler.transform(X_test[:, :, i])

    scalers.append(scaler)

# Optionally save all 3 scalers
joblib.dump(scalers, f'{data_path}/scalers.pkl')


['./scalers.pkl']

In [8]:
# Now you can get num_classes safely
num_classes = len(np.unique(y_all_enc))

# One-hot encode
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

In [9]:
from sklearn.utils import class_weight
import numpy as np

# Compute weights for each class
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dict for Keras
class_weights_dict = dict(enumerate(class_weights_array))


In [10]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dropout
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, BatchNormalization, GlobalAveragePooling1D

input_shape = (X_train_scaled.shape[1], X_train_scaled.shape[2])  # (time_steps, features)

inputs = Input(shape=input_shape)

# 🔍 Feature extractor
x = Conv1D(64, kernel_size=5, activation='relu', padding='same')(inputs)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=2)(x)
x = Dropout(0.2)(x)

x = Conv1D(128, kernel_size=5, activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling1D(pool_size=2)(x)
x = Dropout(0.2)(x)

# 🔁 Sequence modeler
x = Bidirectional(LSTM(64, return_sequences=False))(x)
x = Dropout(0.3)(x)

# 🧠 Classifier head
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])



In [11]:
print(X_train_scaled.shape)  # should be (samples, time_steps, features)
print(np.std(X_train_scaled))  # should not be zero or near-zero


(9088, 500, 3)
1.0000000000000002


In [12]:
print(y_train_cat.shape)  # should be (samples, num_classes)
print(np.unique(np.argmax(y_train_cat, axis=1)))  # Should cover multiple labels


(9088, 62)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61]


In [17]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_cnn_lstm.keras", save_best_only=True, monitor="val_loss", mode="min", verbose=1)


history = model2.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=100,
    batch_size=32,
    callbacks = [early_stop, checkpoint],
    class_weight=class_weights_dict
)


Epoch 1/100
281/284 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.0376 - loss: 3.9947
Epoch 1: val_loss improved from inf to 4.58179, saving model to best_cnn_lstm.keras
284/284 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - accuracy: 0.0379 - loss: 3.9918 - val_accuracy: 0.0403 - val_loss: 4.5818
Epoch 2/100
284/284 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1211 - loss: 3.2930
Epoch 2: val_loss did not improve from 4.58179
284/284 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.1212 - loss: 3.2928 - val_accuracy: 0.0465 - val_loss: 4.8068
Epoch 3/100
283/284 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1744 - loss: 2.8634
Epoch 3: val_loss did not improve from 4.58179
284/284 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.1745 - loss: 2.8633 - val_accuracy: 0.0475 - val_loss: 5.1012
Epoch 4/100
283/284 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.2377 - loss: 2.4971
Epoch 4: val_loss did not improve from 4.58179
284/284 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.2378 - loss: 

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dropout, Dense, Input

model2 = Sequential([
    Input(shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])),

    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    LSTM(64),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
print(X_test_scaled.shape)

(3914, 500, 1)


In [ ]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=1)
print(f"✅ Test accuracy: {test_acc:.2%}")

109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9808 - loss: 0.1009
✅ Test accuracy: 97.88%


In [ ]:
test_2_loss, test_2_acc = model.evaluate(X_reserve_scaled, y_reserve_cat, verbose=1)
print(f"Test Reserve accuracy: {test_2_acc:.2%}")

90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.2911 - loss: 12.9649
Test Reserve accuracy: 18.52%
